In [ ]:
%pip install mp_api

In [ ]:
%pip install ucimlrepo


In [ ]:
%pip install ase


In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

# Fetch dataset (e.g., Polymer / Glass Transition or general benchmark)
dataset = fetch_ucirepo(id=53) # Iris or materials-related tabular dataset
X = dataset.data.features
y = dataset.data.targets

print("Features shape:", X.shape)
print(X.head())

In [ ]:
from google.colab import userdata
from mp_api.client import MPRester

MP_API_KEY = userdata.get('MP_API_KEY')

In [ ]:
from mp_api.client import MPRester
import pandas as pd
import matplotlib.pyplot as plt

# Using MPRester
# Note: Replace 'MP_API_KEY' with your actual key if required
# MP_API_KEY="your-api-key"
with MPRester(api_key = MP_API_KEY) as mpr:
    # Query stable 2D/3D semiconductors with bandgap between 1.0 and 2.0 eV
    docs = mpr.materials.summary.search(
        band_gap=(1.0, 2.0),
        is_stable=True,
        fields=["material_id", "formula_pretty", "band_gap", "formation_energy_per_atom"]
    )

# Convert query results directly to Pandas DataFrame for EDA
data = [{
    "ID": doc.material_id,
    "Formula": doc.formula_pretty,
    "BandGap_eV": doc.band_gap,
    "FormationEnergy_eV": doc.formation_energy_per_atom
} for doc in docs[:50]]

df = pd.DataFrame(data)

# Quick Exploratory Data Analysis (EDA)
plt.figure(figsize=(7, 4))
plt.scatter(df["FormationEnergy_eV"], df["BandGap_eV"], c='teal', alpha=0.7)
plt.xlabel("Formation Energy (eV/atom)")
plt.ylabel("Band Gap (eV)")
plt.title("Band Gap vs Stability (Materials Project)")
plt.grid(True, linestyle="--", alpha=0.1)
plt.show()

In [ ]:
from ase.build import molecule
from pymatgen.core import Structure, Lattice

# Creating an ASE Atoms object (e.g., Water molecule)
water_ase = molecule('H2O')
print("ASE Atom Symbols:", water_ase.get_chemical_symbols())
print("Positions:\n", water_ase.get_positions())


In [ ]:
%pip install rdkit

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

print("🚀 Starting Tg prediction pipeline for Polymers...")

# 1. Mini-Dataset example (Based on SMILES of monomeric units)
# (e.g., Polymethyl methacrylate, Polystyrene, PVC, etc.)
data = {
    "Polymer": ["PMMA", "Polystyrene", "PVC", "Polyethylene", "Polycarbonate"],
    "SMILES": ["*CC(*)(C)C(=O)OC", "*CC(*)(C)c1ccccc1", "*CC(*)Cl", "*CC*", "*Oc1ccc(C(C)(C)c2ccc(O*)cc2)cc1"],
    "Experimental_Tg_K": [378, 373, 354, 148, 420]
}
df = pd.DataFrame(data)

# 2. Feature Engineering with RDKit
# We calculate a couple of basic physicochemical descriptors from the SMILES
def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        mol_weight = Descriptors.MolWt(mol)
        rotatable_bonds = Descriptors.NumRotatableBonds(mol)
        return pd.Series([mol_weight, rotatable_bonds])
    return pd.Series([0, 0])

df[['Molecular_Weight', 'Rotatable_Bonds']] = df['SMILES'].apply(calculate_descriptors)

# 3. Machine Learning: Model Training
# We use the chemical features to predict the Tg
X = df[['Molecular_Weight', 'Rotatable_Bonds']]
y = df['Experimental_Tg_K']

model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X, y)

# 4. Rapid Prediction and Visualization
df['Predicted_Tg_K'] = model.predict(X)

plt.figure(figsize=(6, 4))
plt.scatter(df['Experimental_Tg_K'], df['Predicted_Tg_K'], color='darkorange', s=100)
plt.plot([100, 450], [100, 450], color='gray', linestyle='--') # Ideal line
plt.title("$T_g$ Prediction (Demonstration Model)")
plt.xlabel("Experimental $T_g$ Temperature (K)")
plt.ylabel("Predicted $T_g$ Temperature (K)")
plt.grid(True, alpha=0.3)
plt.show()

print(df[['Polymer', 'Experimental_Tg_K', 'Predicted_Tg_K']])

In [ ]:
# 3. PREDICTION ON NEW DATA (Polypropylene)
new_polymer = "Polypropylene"
new_smiles = "*CC(*)(C)"
new_tg_exp = 260  # Approximate experimental Tg in Kelvin

In [ ]:
features_new = calculate_descriptors(new_smiles)
X_test = pd.DataFrame([features_new.values], columns=['Molecular_Weight', 'Rotatable_Bonds'])
new_tg_pred = model.predict(X_test)[0]

In [ ]:
print(f"--- PREDICTION RESULT ---")
print(f"Polymer: {new_polymer}")
print(f"Experimental Tg: {new_tg_exp} K")
print(f"Predicted Tg:    {new_tg_pred:.1f} K")

In [ ]:
# 4. Visualization with the new point
plt.figure(figsize=(7, 5))

# Training points (Orange)
plt.scatter(df['Experimental_Tg_K'], df['Predicted_Tg_K'],
            color='darkorange', s=100, edgecolors='black', label='Training')

# Test point (Red Star)
plt.scatter(new_tg_exp, new_tg_pred,
            color='crimson', marker='*', s=250, edgecolors='black', label=f'Test: {new_polymer}')

plt.plot([100, 450], [100, 450], color='gray', linestyle='--') # Ideal line
plt.title("$T_g$ Prediction with Inference on New Data")
plt.xlabel("Experimental $T_g$ (K)")
plt.ylabel("Predicted $T_g$ (K)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()